[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/thomas_precession/thomas_precession.ipynb)

# Thomas Precession and the Bargmann–Michel–Telegdi Equation

A spinning relativistic particle carries a unit timelike four-velocity $u$ ($u \cdot u = 1$) and a unit spacelike four-spin $s$ orthogonal to its worldline ($s \cdot u = 0$, $s \cdot s = -1$). An observer at rest along the time axis $e_t$ reads out the particle's rest-frame spin by boosting $u$ back to $e_t$ with the pure boost rotor $(e_t u)^{1/2}$. When the particle accelerates without any torque in its instantaneous rest frame, each small step from $u_{\text{tail}}$ to $u_{\text{head}}$ boosts its frame in the spacetime plane $u_{\text{head}} \wedge u_{\text{tail}}$ by $(u_{\text{head}} u_{\text{tail}})^{1/2}$, discrete Fermi–Walker transport. Because two boosts in different spatial directions do not commute, chaining those pure boosts around a closed circular orbit at Lorentz factor $\gamma = \cosh\eta$ returns the four-velocity to where it started but leaves a spatial turn in the orbital plane by $-2\pi(\gamma - 1)$, the hyperbolic area of the cap the four-velocity sweeps out on the unit hyperboloid: Thomas precession.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.animation import save_figure, show_animation
from examples.relativity.thomas_precession import render
from examples.relativity.thomas_precession.core import circular_orbit_velocities

mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
TIME = mv.t.normalized()                                                        # [] Vector

## 1. Discrete Fermi–Walker transport around a circular orbit

At each step along the circular orbit in the $xy$ plane, the pure boost from `u_tail` to `u_head` is the rotor square root `(u_head * u_tail).square_root()`. Carrying the initial spin along the loop by `turn >> spin` keeps `loop_u | loop_s == 0` at every step, while the laboratory spin `(TIME * loop_u).square_root() >> loop_s` turns clockwise by $2\pi(\gamma - 1)$ per orbit: $72^\circ$ at $\gamma = 1.2$ on the inner teal ring, $180^\circ$ at $\gamma = 1.5$ on the middle orange ring, and a full $360^\circ$ turn at $\gamma = 2.0$ on the outer blue ring.

In [ ]:
GAMMAS = np.array([1.2, 1.5, 2.0])                                              # [orbits]
LOOP_STEPS = 180

rapidities = np.arccosh(GAMMAS)                                                 # [orbits]
loop_pos, loop_u, start_spins = circular_orbit_velocities(rapidities, LOOP_STEPS)

# Pure boost from each four-velocity along the orbit to the next.
boost_steps = (loop_u[1:] * loop_u[:-1]).square_root()                          # [steps, orbits] Rotor
carried = [start_spins]
for step in boost_steps:
    carried.append(step >> carried[-1])                                         # [orbits] Vector
loop_s = stack(carried)                                                         # [steps + 1, orbits] Vector
loop_lab = (TIME * loop_u).square_root() >> loop_s                              # [steps + 1, orbits] Vector

## 2. The Bargmann–Michel–Telegdi generator in a storage ring

In an electromagnetic field bivector `field`, with unit charge-to-mass ratio, the Lorentz-force four-acceleration is `acceleration = field | u`. For a particle with gyromagnetic ratio `g`, the single spacetime bivector

$$\Omega = \frac{g}{2} F - \left(\frac{g}{2} - 1\right)(a \wedge u)$$

generates both the orbit and the spin precession. Because `u | u == 1` and `acceleration | u == 0`, the interior product `(acceleration ^ u) | u` equals `acceleration`, so `omega | u == field | u` regardless of `g`. On any four-spin `s` orthogonal to `u`, `(acceleration ^ u) | s == -(acceleration | s) * u`, so the linear map `omega | Vector` (`Vector <- Vector`) is the Bargmann–Michel–Telegdi rate map. Stepping a single Lorentz rotor `frame` by `(omega * (dt / 2)).exp() * frame` evolves both `u = frame >> TIME` and `s = frame >> rest_spin` at once. For a Dirac particle ($g = 2$, orange) the acceleration wedge drops out and $\Omega = F$, locking the spin to the velocity around a magnetic storage ring; for a torque-free gyroscope ($g = 0$, teal) only the Fermi–Walker term $-a \wedge u$ remains; for an anomalous magnetic moment ($g = 2.5$, blue) the spin precesses ahead of the velocity at rate $\gamma(g/2 - 1)$ per cyclotron turn.

In the four-tensor index notation of relativistic electrodynamics, the field bivector reads as the antisymmetric tensor $F^{\mu\nu}$ and the Bargmann–Michel–Telegdi equation as $\frac{ds^\mu}{d\tau} = \frac{q}{m}\left(\frac{g}{2} F^{\mu}{}_{\nu} s^\nu + \left(\frac{g}{2} - 1\right) u^\mu F_{\nu\lambda} u^\nu s^\lambda\right)$.

In [ ]:
RING_GAMMA = 1.6
G_FACTORS = np.array([0.0, 2.0, 2.5])                                           # [cases]
CYCLOTRON_RATE = 1.0
RING_STEPS = 180
EVERY = 6


def bmt_generator(field: Bivector, four_velocity: Vector, g_factor: np.ndarray) -> Bivector:
    """The spacetime bivector generating both the Lorentz-force acceleration of `four_velocity` and
    the Bargmann–Michel–Telegdi precession of any four-spin orthogonal to it."""
    acceleration = field | four_velocity                                        # [...] Vector
    half_g = g_factor / 2.0
    return half_g * field - (half_g - 1.0) * (acceleration ^ four_velocity)     # [...] Bivector


ring_rapidities = np.full_like(G_FACTORS, np.arccosh(RING_GAMMA))               # [cases]
ring_pos, orbit_u, _ = circular_orbit_velocities(ring_rapidities, RING_STEPS)
field = CYCLOTRON_RATE * mv.xy                                                  # [] Bivector
dt = 2 * np.pi / (CYCLOTRON_RATE * RING_STEPS)
start_frame = (orbit_u[0] * TIME).square_root()                                 # [cases] Rotor
rest_spin = mv.y                                                                # [] Vector

frames = [start_frame]
for _ in range(RING_STEPS):
    current = frames[-1]                                                        # [cases] Rotor
    omega_start = bmt_generator(field, current >> TIME, G_FACTORS)              # [cases] Bivector
    halfway = (omega_start * (dt / 4.0)).exp() * current                        # [cases] Rotor
    omega_mid = bmt_generator(field, halfway >> TIME, G_FACTORS)                # [cases] Bivector
    frames.append((omega_mid * (dt / 2.0)).exp() * current)                     # [cases] Rotor
history = stack(frames)                                                         # [steps + 1, cases] Rotor
ring_u = history >> TIME                                                        # [steps + 1, cases] Vector
ring_s = history >> rest_spin                                                   # [steps + 1, cases] Vector
ring_lab = (TIME * ring_u).square_root() >> ring_s                              # [steps + 1, cases] Vector

save_figure(render.draw_thomas_summary(loop_pos, loop_u, loop_lab, ring_pos, ring_lab), "thomas_precession");

In [ ]:
show_animation(render.animate_storage_ring(ring_pos, ring_lab, EVERY), "thomas_precession", 50)

In [ ]:
# checks
np.testing.assert_allclose((loop_u | loop_u).kernel, 1.0, atol=1e-12)
np.testing.assert_allclose((loop_s | loop_s).kernel, -1.0, atol=1e-12)
np.testing.assert_allclose((loop_u | loop_s).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((loop_lab | TIME).kernel, 0.0, atol=1e-12)

expected_loop_end = ((mv.xy * (-np.pi * (GAMMAS - 1.0))).exp()) >> start_spins
np.testing.assert_allclose((loop_lab[-1] - expected_loop_end).kernel, 0.0, atol=5e-3)

np.testing.assert_allclose((ring_u | ring_s).kernel, 0.0, atol=1e-12)
net_angles = 2 * np.pi * (1.0 + RING_GAMMA * (G_FACTORS / 2.0 - 1.0))
expected_ring_end = ((mv.xy * (net_angles / 2.0)).exp()) >> rest_spin
np.testing.assert_allclose((ring_lab[-1] - expected_ring_end).kernel, 0.0, atol=1e-2)